In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.models.batter_score import build_profiles, qualified
from src.data.player_ids import load_player_ids, display_name

df = load_all_snapshots(seasons=[2024])
prof = build_profiles(df)
q = qualified(prof)

ids = load_player_ids(q.index.tolist())
q = q.join(display_name(ids))
print(f"{len(q)} qualified batters")
print(q.columns.tolist())

343 qualified batters
['pitches', 'zone_pct', 'swing_pct', 'chase_pct', 'zone_swing_pct', 'contact_pct', 'zone_contact_pct', 'whiff_pct', 'n_out_of_zone', 'n_in_zone', 'n_swings', 'n_zone_swings', 'bbe', 'barrel_pct', 'hard_hit_pct', 'sweet_spot_pct', 'avg_exit_velocity', 'max_exit_velocity', 'avg_launch_angle', 'woba', 'name']


In [2]:
CANDIDATES = ["chase_pct", "zone_swing_pct", "zone_contact_pct",
              "barrel_pct", "hard_hit_pct", "sweet_spot_pct",
              "avg_exit_velocity", "avg_launch_angle", "whiff_pct"]

corr = q[CANDIDATES].corr()
print(corr.round(2).to_string())
print()

# Which pairs exceed the Day 16 redundancy threshold?
pairs = []
for i, a in enumerate(CANDIDATES):
    for b in CANDIDATES[i+1:]:
        r = corr.loc[a, b]
        if abs(r) > 0.7:
            pairs.append((a, b, round(r, 3)))
print("pairs above |r| = 0.7:")
for a, b, r in pairs:
    print(f"  {a} / {b}: {r}")

                   chase_pct  zone_swing_pct  zone_contact_pct  barrel_pct  hard_hit_pct  sweet_spot_pct  avg_exit_velocity  avg_launch_angle  whiff_pct
chase_pct               1.00            0.53             -0.04       -0.11         -0.12           -0.10              -0.16             -0.15       0.18
zone_swing_pct          0.53            1.00             -0.30        0.14          0.11            0.13               0.08              0.05       0.36
zone_contact_pct       -0.04           -0.30              1.00       -0.54         -0.36           -0.12              -0.28             -0.16      -0.92
barrel_pct             -0.11            0.14             -0.54        1.00          0.81            0.34               0.77              0.33       0.56
hard_hit_pct           -0.12            0.11             -0.36        0.81          1.00            0.18               0.93              0.06       0.41
sweet_spot_pct         -0.10            0.13             -0.12        0.34        

In [3]:
# Keep one per redundant group. Reasoning mirrors Day 16 and Day 36.
SIMILARITY_FEATURES = [
    "chase_pct",          # plate discipline
    "zone_contact_pct",   # bat-to-ball (whiff_pct is its near-complement)
    "barrel_pct",         # power (hard_hit_pct is r=0.78 redundant)
    "avg_launch_angle",   # batted ball profile: ground vs air
]

print(q[SIMILARITY_FEATURES].corr().round(2).to_string())
print()
print(q[SIMILARITY_FEATURES].describe().round(3).to_string())

                  chase_pct  zone_contact_pct  barrel_pct  avg_launch_angle
chase_pct              1.00             -0.04       -0.11             -0.15
zone_contact_pct      -0.04              1.00       -0.54             -0.16
barrel_pct            -0.11             -0.54        1.00              0.33
avg_launch_angle      -0.15             -0.16        0.33              1.00

       chase_pct  zone_contact_pct  barrel_pct  avg_launch_angle
count    343.000           343.000     343.000           343.000
mean       0.285             0.853       0.078            13.163
std        0.056             0.047       0.040             4.569
min        0.156             0.699       0.007            -0.215
25%        0.247             0.823       0.049            10.274
50%        0.281             0.856       0.073            13.324
75%        0.323             0.886       0.103            16.250
max        0.460             0.966       0.270            26.475


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity

X = StandardScaler().fit_transform(q[SIMILARITY_FEATURES])

euc = pd.DataFrame(euclidean_distances(X), index=q.index, columns=q.index)
cos = pd.DataFrame(cosine_similarity(X), index=q.index, columns=q.index)

def most_similar(batter_id, matrix, n=8, ascending=True):
    s = matrix.loc[batter_id].drop(batter_id)
    picks = s.nsmallest(n) if ascending else s.nlargest(n)
    out = q.loc[picks.index, ["name"] + SIMILARITY_FEATURES].copy()
    out.insert(1, "score", picks.round(3))
    return out

JUDGE = 592450
print("=== Judge, by Euclidean distance ===")
print(most_similar(JUDGE, euc).round(3).to_string(index=False))
print()
print("=== Judge, by cosine similarity ===")
print(most_similar(JUDGE, cos, ascending=False).round(3).to_string(index=False))
print()
print("Judge's own profile:")
print(q.loc[JUDGE, SIMILARITY_FEATURES].round(3).to_string())

=== Judge, by Euclidean distance ===
              name  score  chase_pct  zone_contact_pct  barrel_pct  avg_launch_angle
    Ohtani, Shohei  2.012      0.259             0.803       0.218            16.368
    O'Neill, Tyler  2.861      0.257             0.769       0.173            20.169
        Soto, Juan  2.927      0.178             0.855       0.198            10.102
Stanton, Giancarlo  2.935      0.309             0.779       0.209            14.755
   Schwarber, Kyle  3.011      0.207             0.792       0.156            15.036
   Toglia, Michael  3.036      0.270             0.796       0.173            15.114
    Ozuna, Marcell  3.346      0.263             0.800       0.157            14.553
     Rooker, Brent  3.443      0.306             0.811       0.167            18.935

=== Judge, by cosine similarity ===
             name  score  chase_pct  zone_contact_pct  barrel_pct  avg_launch_angle
   Ohtani, Shohei  0.971      0.259             0.803       0.218            

In [5]:
# If similarity is meaningful, similar players should have similar wOBA.
def neighbor_woba(matrix, k=5, ascending=True):
    preds = {}
    for bid in q.index:
        s = matrix.loc[bid].drop(bid)
        nb = s.nsmallest(k).index if ascending else s.nlargest(k).index
        preds[bid] = q.loc[nb, "woba"].mean()
    return pd.Series(preds)

pred_euc = neighbor_woba(euc, ascending=True)
pred_cos = neighbor_woba(cos, ascending=False)

print("correlation between k=5 neighbour wOBA and own wOBA:")
print(f"  euclidean: {pred_euc.corr(q['woba']):.3f}")
print(f"  cosine:    {pred_cos.corr(q['woba']):.3f}")
print(f"  (baseline: league mean predicts nothing, r = 0)")
print()
print("MAE:")
print(f"  euclidean: {(pred_euc - q['woba']).abs().mean():.4f}")
print(f"  cosine:    {(pred_cos - q['woba']).abs().mean():.4f}")
print(f"  league:    {(q['woba'].mean() - q['woba']).abs().mean():.4f}")

correlation between k=5 neighbour wOBA and own wOBA:


TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'

In [6]:
def neighbor_woba(matrix, k=5, ascending=True):
    """Mean wOBA of the k nearest neighbours.

    Build the Series with q.index directly. Constructing it from a dict
    keyed by Int64 labels produces an index pandas cannot align.
    """
    vals = []
    for bid in q.index:
        s = matrix.loc[bid].drop(bid)
        nb = s.nsmallest(k).index if ascending else s.nlargest(k).index
        vals.append(pd.to_numeric(q.loc[nb, "woba"], errors="coerce").mean())
    return pd.Series(vals, index=q.index, dtype="float64")

woba = pd.to_numeric(q["woba"], errors="coerce").astype("float64")

pred_euc = neighbor_woba(euc, ascending=True)
pred_cos = neighbor_woba(cos, ascending=False)

print("correlation between k=5 neighbour wOBA and own wOBA:")
print(f"  euclidean: {pred_euc.corr(woba):.3f}")
print(f"  cosine:    {pred_cos.corr(woba):.3f}")
print()
print("MAE:")
print(f"  euclidean: {(pred_euc - woba).abs().mean():.4f}")
print(f"  cosine:    {(pred_cos - woba).abs().mean():.4f}")
print(f"  league:    {(woba.mean() - woba).abs().mean():.4f}")

correlation between k=5 neighbour wOBA and own wOBA:
  euclidean: 0.567
  cosine:    0.538

MAE:
  euclidean: 0.0239
  cosine:    0.0246
  league:    0.0285


In [7]:
# Does any single feature dominate the distance?
for feat in SIMILARITY_FEATURES:
    others = [f for f in SIMILARITY_FEATURES if f != feat]
    Xo = StandardScaler().fit_transform(q[others])
    euc_o = pd.DataFrame(euclidean_distances(Xo), index=q.index, columns=q.index)
    top_full = set(euc.loc[JUDGE].drop(JUDGE).nsmallest(8).index)
    top_less = set(euc_o.loc[JUDGE].drop(JUDGE).nsmallest(8).index)
    print(f"drop {feat:20s} -> {len(top_full & top_less)}/8 neighbours retained")

drop chase_pct            -> 6/8 neighbours retained
drop zone_contact_pct     -> 7/8 neighbours retained
drop barrel_pct           -> 1/8 neighbours retained
drop avg_launch_angle     -> 8/8 neighbours retained


In [8]:
# Is barrel dominance universal, or a Judge-specific artifact?
import random
random.seed(42)
sample = random.sample(list(q.index), 20)

retention = {f: [] for f in SIMILARITY_FEATURES}
for pid in sample:
    top_full = set(euc.loc[pid].drop(pid).nsmallest(8).index)
    for feat in SIMILARITY_FEATURES:
        others = [f for f in SIMILARITY_FEATURES if f != feat]
        Xo = StandardScaler().fit_transform(q[others])
        e = pd.DataFrame(euclidean_distances(Xo), index=q.index, columns=q.index)
        top_less = set(e.loc[pid].drop(pid).nsmallest(8).index)
        retention[feat].append(len(top_full & top_less))

print("mean neighbours retained when dropping each feature (20 random batters):")
for f, vals in retention.items():
    print(f"  {f:20s} {np.mean(vals):.1f}/8")

mean neighbours retained when dropping each feature (20 random batters):
  chase_pct            3.9/8
  zone_contact_pct     3.9/8
  barrel_pct           5.0/8
  avg_launch_angle     3.2/8
